# PROJECT 2: CUSTOMER CHURN PREDICTION ANALYSIS
## Telecom Industry - Predictive Analytics & Retention Strategy

**Objective**: Identify at-risk customers before they churn using machine learning

**Timeline**: 6 weeks | **Skill Level**: Mid-level | **Tools**: Python + Pandas + Scikit-learn

---

## TASK 1: DATA MANIPULATION
### Extract columns, filter segments, verify data quality

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, roc_curve, mean_squared_error
)
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

### 1A: Load Data

In [ ]:
# Load the synthetic customer churn dataset
churn_df = pd.read_csv(r'c:\Users\Administrator\Desktop\project\p2\customer_churn_synthetic.csv')

print(f"Dataset shape: {churn_df.shape}")
print(f"\nFirst few rows:")
print(churn_df.head())

print(f"\nData types:")
print(churn_df.dtypes)

print(f"\nMissing values:")
print(churn_df.isnull().sum())

print(f"\nBasic statistics:")
print(churn_df.describe())

### 1B: Extract Specific Columns

In [ ]:
# Subtask 1B: Extract specific columns

# Extract 5th column
customer_5 = churn_df.iloc[:, 4]
print(f"5th column (Partner): {customer_5.head()}")

# Extract 15th column
customer_15 = churn_df.iloc[:, 14]
print(f"\n15th column (Contract): {customer_15.head()}")

### 1C: Filter Customer Segments

In [ ]:
# Extract customers with tenure > 70 months OR monthly charges > $100
customer_total_tenure = churn_df[(churn_df['tenure'] > 70) | (churn_df['MonthlyCharges'] > 100)]
print(f"Customers with tenure > 70 months OR MonthlyCharges > $100: {len(customer_total_tenure)}")
print(customer_total_tenure.head())

### 1D: Churn Distribution

In [ ]:
# Get count of different churn levels
churn_counts = churn_df['Churn'].value_counts()
print("\n=== CHURN DISTRIBUTION ===")
print(churn_counts)
print(f"\nChurn Rate: {(churn_df['Churn']=='Yes').mean():.2%}")

---
## TASK 2: DATA VISUALIZATION
### Create visualizations for key features and relationships

### 2A: Bar Plot - Internet Service Distribution

In [ ]:
# Bar plot for InternetService
fig, ax = plt.subplots(figsize=(10, 6))
internet_counts = churn_df['InternetService'].value_counts()
internet_counts.plot(kind='bar', ax=ax, color='orange', edgecolor='black')
ax.set_xlabel('Categories of Internet Service')
ax.set_ylabel('Count of Categories')
ax.set_title('Distribution of Internet Service')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

### 2B: Histogram - Tenure Distribution

In [ ]:
# Histogram for tenure
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(churn_df['tenure'], bins=30, color='green', edgecolor='black')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of tenure')
plt.tight_layout()
plt.show()

### 2C: Scatter Plot - Tenure vs Monthly Charges

In [ ]:
# Scatter plot: tenure vs MonthlyCharges
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(churn_df['tenure'], churn_df['MonthlyCharges'], color='brown', alpha=0.5)
ax.set_xlabel('Tenure of customer')
ax.set_ylabel('Monthly Charges of customer')
ax.set_title('Tenure vs Monthly Charges')
plt.tight_layout()
plt.show()

### 2D: Box Plot - Tenure by Contract Type

In [ ]:
# Box plot: tenure by Contract
fig, ax = plt.subplots(figsize=(10, 6))
churn_df.boxplot(column='tenure', by='Contract', ax=ax)
ax.set_xlabel('Contract Type')
ax.set_ylabel('Tenure (months)')
ax.set_title('Tenure by Contract Type')
plt.suptitle('')  # Remove the default title
plt.tight_layout()
plt.show()

---
## TASK 3: LINEAR REGRESSION
### Predict Monthly Charges based on Tenure

In [ ]:
# Prepare data for linear regression
X_lr = churn_df[['tenure']]
y_lr = churn_df['MonthlyCharges']

# Split data (70:30)
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.30, random_state=42
)

print(f"Training set size: {X_train_lr.shape[0]}")
print(f"Test set size: {X_test_lr.shape[0]}")

In [ ]:
# Build and train linear regression model
model_lr = LinearRegression()
model_lr.fit(X_train_lr, y_train_lr)

# Make predictions
y_pred_lr = model_lr.predict(X_test_lr)

# Calculate error and RMSE
error = y_test_lr - y_pred_lr
rmse = np.sqrt(mean_squared_error(y_test_lr, y_pred_lr))

print(f"\n=== LINEAR REGRESSION RESULTS ===")
print(f"Prediction Error (first 5): {error.head().values}")
print(f"Root Mean Square Error (RMSE): {rmse:.2f}")

In [ ]:
# Visualize linear regression
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_test_lr, y_test_lr, color='blue', alpha=0.5, label='Actual')
ax.plot(X_test_lr, y_pred_lr, color='red', linewidth=2, label='Predicted')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Monthly Charges ($)')
ax.set_title(f'Linear Regression: Tenure vs Monthly Charges (RMSE: {rmse:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

---
## TASK 4: LOGISTIC REGRESSION
### Simple and Multiple Logistic Regression Models for Churn Prediction

### 4A: Simple Logistic Regression (MonthlyCharges only)

In [ ]:
# Prepare data for simple logistic regression
churn_df['Churn_binary'] = (churn_df['Churn'] == 'Yes').astype(int)

X_logistic = churn_df[['MonthlyCharges']]
y_logistic = churn_df['Churn_binary']

# Split (65:35)
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_logistic, y_logistic, test_size=0.35, random_state=42, stratify=y_logistic
)

# Train simple logistic regression
model_log_simple = LogisticRegression(random_state=42, max_iter=1000)
model_log_simple.fit(X_train_log, y_train_log)

# Predictions
y_pred_log_simple = model_log_simple.predict(X_test_log)

# Evaluate
cm_simple = confusion_matrix(y_test_log, y_pred_log_simple)
accuracy_simple = accuracy_score(y_test_log, y_pred_log_simple)

print("=== SIMPLE LOGISTIC REGRESSION (MonthlyCharges) ===")
print(f"Accuracy: {accuracy_simple:.4f}")
print(f"\nConfusion Matrix:")
print(cm_simple)
print(f"\nClassification Report:")
print(classification_report(y_test_log, y_pred_log_simple, target_names=['No Churn', 'Churn']))

### 4B: Multiple Logistic Regression (Tenure + MonthlyCharges)

In [ ]:
# Prepare data for multiple logistic regression
X_logistic_multi = churn_df[['tenure', 'MonthlyCharges']]
y_logistic_multi = churn_df['Churn_binary']

# Split (80:20)
X_train_log_m, X_test_log_m, y_train_log_m, y_test_log_m = train_test_split(
    X_logistic_multi, y_logistic_multi, test_size=0.20, random_state=42, stratify=y_logistic_multi
)

# Train multiple logistic regression
model_log_multi = LogisticRegression(random_state=42, max_iter=1000)
model_log_multi.fit(X_train_log_m, y_train_log_m)

# Predictions
y_pred_log_multi = model_log_multi.predict(X_test_log_m)

# Evaluate
cm_multi = confusion_matrix(y_test_log_m, y_pred_log_multi)
accuracy_multi = accuracy_score(y_test_log_m, y_pred_log_multi)

print("\n=== MULTIPLE LOGISTIC REGRESSION (Tenure + MonthlyCharges) ===")
print(f"Accuracy: {accuracy_multi:.4f}")
print(f"\nConfusion Matrix:")
print(cm_multi)
print(f"\nClassification Report:")
print(classification_report(y_test_log_m, y_pred_log_multi, target_names=['No Churn', 'Churn']))

---
## TASK 5: DECISION TREE
### Churn Prediction using Tenure

In [ ]:
# Prepare data for decision tree
X_dt = churn_df[['tenure']]
y_dt = churn_df['Churn_binary']

# Split (80:20)
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X_dt, y_dt, test_size=0.20, random_state=42, stratify=y_dt
)

# Train decision tree
model_dt = DecisionTreeClassifier(random_state=42, max_depth=10)
model_dt.fit(X_train_dt, y_train_dt)

# Predictions
y_pred_dt = model_dt.predict(X_test_dt)

# Evaluate
cm_dt = confusion_matrix(y_test_dt, y_pred_dt)
accuracy_dt = accuracy_score(y_test_dt, y_pred_dt)

print("=== DECISION TREE (Tenure) ===")
print(f"Accuracy: {accuracy_dt:.4f}")
print(f"\nConfusion Matrix:")
print(cm_dt)
print(f"\nClassification Report:")
print(classification_report(y_test_dt, y_pred_dt, target_names=['No Churn', 'Churn']))

---
## TASK 6: RANDOM FOREST
### Advanced Multi-Feature Churn Prediction

In [ ]:
# Prepare data for random forest
X_rf = churn_df[['tenure', 'MonthlyCharges']]
y_rf = churn_df['Churn_binary']

# Split (70:30)
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, test_size=0.30, random_state=42, stratify=y_rf
)

# Train random forest
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
model_rf.fit(X_train_rf, y_train_rf)

# Predictions
y_pred_rf = model_rf.predict(X_test_rf)

# Evaluate
cm_rf = confusion_matrix(y_test_rf, y_pred_rf)
accuracy_rf = accuracy_score(y_test_rf, y_pred_rf)

print("=== RANDOM FOREST (Tenure + MonthlyCharges) ===")
print(f"Accuracy: {accuracy_rf:.4f}")
print(f"\nConfusion Matrix:")
print(cm_rf)
print(f"\nClassification Report:")
print(classification_report(y_test_rf, y_pred_rf, target_names=['No Churn', 'Churn']))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': ['tenure', 'MonthlyCharges'],
    'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
print(feature_importance)

---
## MODEL COMPARISON SUMMARY

In [ ]:
# Compare all models
model_comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression (Simple)',
        'Logistic Regression (Multiple)',
        'Decision Tree',
        'Random Forest'
    ],
    'Test Accuracy': [
        accuracy_simple,
        accuracy_multi,
        accuracy_dt,
        accuracy_rf
    ]
})

print("\n" + "=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
print(model_comparison.to_string(index=False))
print(f"\n🏆 Best Model: {model_comparison.loc[model_comparison['Test Accuracy'].idxmax(), 'Model']}")
print(f"   Accuracy: {model_comparison['Test Accuracy'].max():.4f}")

---
## PROJECT COMPLETION SUMMARY

In [ ]:
print("\n" + "=" * 70)
print("✅ PROJECT 2: CUSTOMER CHURN PREDICTION - COMPLETE")
print("=" * 70)

print("\n📊 TASKS COMPLETED:")
print("✅ TASK 1: Data Manipulation (Extract & Filter)")
print("✅ TASK 2: Data Visualization (4 charts)")
print("✅ TASK 3: Linear Regression (Tenure → MonthlyCharges)")
print("✅ TASK 4: Logistic Regression (Simple & Multiple)")
print("✅ TASK 5: Decision Tree (Tenure predictor)")
print("✅ TASK 6: Random Forest (Multi-feature model)")

print("\n📈 KEY INSIGHTS:")
print(f"  • Dataset: {churn_df.shape[0]:,} customers, {churn_df.shape[1]} features")
print(f"  • Churn Rate: {(churn_df['Churn']=='Yes').mean():.1%}")
print(f"  • Best Model: Random Forest ({accuracy_rf:.2%} accuracy)")
print(f"  • Most Important Feature: {feature_importance.iloc[0]['feature']} ({feature_importance.iloc[0]['importance']:.2%})")

print("\n" + "=" * 70)